# Globally-Fitting (with Replicas) Experimental Data as Supervised Learning

## (1): Import Libraries:

In [ ]:
import datetime
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## (2): Matplotlib rcparams Configuration:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Loading Data:

### (3.1): Loading Main File:

In [ ]:
test_dataframe = pd.read_csv('../data/burner_data.csv')

### (3.2): Loading in the supervised learning $(x, y)$ pairs:

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]].copy()
y_data = test_dataframe[["unp_beam_unp_target_xsec", "unp_target_bsa"]].copy()

### (3.3): Checking out the $x$ data:

In [ ]:
x_data.head()

### (3.4): Checking out the $y$ data:

In [ ]:
y_data.head()

## (4): Data Preprocessing:

Lot of good resources on this already, but it is indeed a bit of a trivial data manipulation. Still, it is helpful to have resources:
1. https://medium.com/@WojtekFulmyk/standardizing-data-for-machine-learning-2cf687e621f9

### (4.1): Finding Statistics of the Raw Labels:

In [ ]:
raw_data_t_max = x_data['t'].max()
raw_data_t_min = x_data['t'].min()
raw_data_t_average = x_data['t'].mean()
raw_data_t_stddev = x_data['t'].std()

print(f"[INFO]: Raw data -t is bound between: {raw_data_t_min} and {raw_data_t_max}")
print(f"[INFO]: Raw data -t has mean = {raw_data_t_average} and stddev = {raw_data_t_stddev}")

x_data['minus_t'] = -x_data['t']

raw_data_minus_t_max = x_data['minus_t'].max()
raw_data_minus_t_min = x_data['minus_t'].min()
raw_data_minus_t_average = x_data['minus_t'].mean()
raw_data_minus_t_stddev = x_data['minus_t'].std()

print(f"[INFO]: Raw data t is bound between: {raw_data_minus_t_min} and {raw_data_minus_t_max}")
print(f"[INFO]: Raw data t has mean = {raw_data_minus_t_average} and stddev = {raw_data_minus_t_stddev}")

raw_data_xb_max = x_data['x_b'].max()
raw_data_xb_min = x_data['x_b'].min()
raw_data_xb_average = x_data['x_b'].mean()
raw_data_xb_stddev = x_data['x_b'].std()

print(f"[INFO]: Raw data xb is bound between: {raw_data_xb_min} and {raw_data_xb_max}")
print(f"[INFO]: Raw data xb has mean = {raw_data_xb_average} and stddev = {raw_data_xb_stddev}")

raw_data_qsq_max = x_data['q_squared'].max()
raw_data_qsq_min = x_data['q_squared'].min()
raw_data_qsq_average = x_data['q_squared'].mean()
raw_data_qsq_stddev = x_data['q_squared'].std()

print(f"[INFO]: Raw data Q^2 is bound between: {raw_data_qsq_min} and {raw_data_qsq_max}")
print(f"[INFO]: Raw data Q^2 has mean = {raw_data_qsq_average} and stddev = {raw_data_qsq_stddev}")

### (4.2): Beginning to Standardize Data:

#### (4.2.1): Standardizing Feature Data:

In [ ]:
# kinematics standardization:
x_data["xb_standardized"] = ((x_data["x_b"] - raw_data_xb_average) /  raw_data_xb_stddev)
x_data["qsquared_standardized"] = ((x_data["q_squared"] - raw_data_qsq_average) /  raw_data_qsq_stddev)

x_data["minus_t_standardized"] = ((x_data["minus_t"] - raw_data_minus_t_average) /  raw_data_minus_t_stddev)

# phi "standardization":
x_data["phi_cos"] = np.cos(x_data["phi"])
x_data["phi_sin"] = np.sin(x_data["phi"])

In [ ]:
def build_features(raw_dataframe):
    features = pd.DataFrame(index = raw_dataframe.index)

    # kinematics standardization:
    features["xb_standardized"] = (raw_dataframe["x_b"] - raw_data_xb_average) / raw_data_xb_stddev
    features["qsquared_standardized"] = (raw_dataframe["q_squared"] - raw_data_qsq_average) / raw_data_qsq_stddev
    features["minus_t_standardized"] = (raw_dataframe["minus_t"] - raw_data_minus_t_average) / raw_data_minus_t_stddev

    # phi "standardization":
    features["phi_cos"] = np.cos(raw_dataframe["phi"])
    features["phi_sin"] = np.sin(raw_dataframe["phi"])

    return features

#### (4.2.2): Standardizing Label Data:

In [ ]:
raw_data_cross_section_average = y_data['unp_beam_unp_target_xsec'].mean()
raw_data_cross_section_stddev = y_data['unp_beam_unp_target_xsec'].std()
raw_data_bsa_average = y_data['unp_target_bsa'].mean()
raw_data_bsa_stddev = y_data['unp_target_bsa'].std()

print(f"[INFO]: Raw data cross-section has mean = {raw_data_cross_section_average} and stddev = {raw_data_cross_section_stddev}")
print(f"[INFO]: Raw data BSA has mean = {raw_data_bsa_average} and stddev = {raw_data_bsa_stddev}")

In [ ]:
def build_labels(raw_dataframe):
    labels = pd.DataFrame(index = raw_dataframe.index)

    labels["unp_beam_unp_target_xsec_standardized"] = ((y_data["unp_beam_unp_target_xsec"] - raw_data_cross_section_average) /  raw_data_cross_section_stddev)
    labels["unp_target_bsa_standardized"] = ((y_data["unp_target_bsa"] - raw_data_bsa_average) /  raw_data_bsa_stddev)

    return labels

### (4.3): Now redefine the training data:

In [ ]:
features = build_features(x_data)
labels = build_labels(y_data)

### (4.4): Checking out the standarized $x$ data:

In [ ]:
features.head()

### (3.5): Splitting along training/validation/testing:

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    features, labels,
    test_size = 0.1, shuffle = True, random_state = 42)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True, random_state = 42)

In [ ]:
print(f"[INFO]: Training points = {len(x_training)}, validation points = {len(x_validation)}, testing points = {len(x_testing)}")

## (4): Defining the DNN Model:

### (4.1): MSE Loss:

In [ ]:
class MSELoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred))

### (4.2): DNN Architecture:

In [ ]:
class SurrogateModel(tf.keras.Model):

    # https://keras.io/api/models/model/ -> follow this for custom model architecture

    def __init__(self, cross_section_symmetry_weight = 0.5, bsa_symmetry_weight = 0.5):
        super().__init__()

        self.cross_section_symmetry_weight = cross_section_symmetry_weight
        self.bsa_symmetry_weight = bsa_symmetry_weight

        # regularizer:
        # I learned about regularizers here: https://medium.com/@theo.wolf/physics-informed-neural-networks-a-simple-tutorial-with-pytorch-f28a890b874a
        # weight_regularizer = regularizers.l2(0.01) 

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_2 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_3 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(2, activation = "linear", name = "cross_section")

        # custom loss business:
        self.cross_section_loss_tracker = MSELoss()

    def azimuthal_symmetry_loss(self, X_batch, training = True):

        X_plus = X_batch

        phi_cos = X_batch[:, 3]
        phi_sin = X_batch[:, 4]

        X_minus = tf.stack([X_batch[:, 0], X_batch[:, 1], X_batch[:, 2], phi_cos, -phi_sin], axis = 1)

        y_plus = self(X_plus, training = training)[:, 0]
        y_minus = self(X_minus, training = training)[:, 0]

        return tf.reduce_mean(tf.square(y_plus - y_minus))
    
    def bsa_azimuthal_symmetry_loss(self, X_batch, training = True):

        X_plus = X_batch

        phi_cos = X_batch[:, 3]
        phi_sin = X_batch[:, 4]

        X_minus = tf.stack([X_batch[:, 0], X_batch[:, 1],X_batch[:, 2],phi_cos, -phi_sin], axis = 1)

        y_plus = self(X_plus, training=training)[:, 1]
        y_minus = self(X_minus, training=training)[:, 1]

        return tf.reduce_mean(tf.square(y_plus + y_minus))

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)
        cross_section_output = self.cross_section_output(hidden_layer)

        return cross_section_output
    
    def train_step(self, data):

        # unpack data:
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            # forward pass:
            predictions = self(X_batch_data, training = True)

            # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
            data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

            # compute phi symmetry loss:
            cross_section_symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

            # compute BSA(phi) symmetry loss:
            bsa_symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = True)

            # total loss is just a weighted sum:
            total_loss = (
                data_loss + 
                self.cross_section_symmetry_weight * cross_section_symmetry_loss +
                self.bsa_symmetry_weight * bsa_symmetry_loss
                )

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.trainable_variables))

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "cross_section_symmetry_loss": cross_section_symmetry_loss,
            "bsa_symmetry_loss": bsa_symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

    def test_step(self, data):
        # unpack data:
        X_batch_data, y_batch_data = data
        
        # forward pass evaluation:
        predictions = self(X_batch_data, training = False)

        # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
        data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

         # compute phi symmetry loss:
        cross_section_symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

        # compute BSA(phi) symmetry loss:
        bsa_symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = True)
        
        # total loss is just a weighted sum:
        total_loss = (
                data_loss + 
                self.cross_section_symmetry_weight * cross_section_symmetry_loss +
                self.bsa_symmetry_weight * bsa_symmetry_loss
                )

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)
            
        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "cross_section_symmetry_loss": cross_section_symmetry_loss,
            "bsa_symmetry_loss": bsa_symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

## (5): **The Main Part of the Program: Replica Method!**

### (5.1): Inverse Transform function for the Label Data:

In [ ]:
def inverse_transform_labels(predictions):

    dnn_predictions = np.array(predictions)
    physical_labels = np.empty_like(predictions)

    # inverse transform
    physical_labels[:, 0] = dnn_predictions[:, 0] * raw_data_cross_section_stddev + raw_data_cross_section_average
    physical_labels[:, 1] = dnn_predictions[:, 1] * raw_data_bsa_stddev + raw_data_bsa_average

    return physical_labels

### (5.2): **Replica Method**:

In [ ]:
NUMBER_OF_EPOCHS = 750
number_of_replicas = 2

all_histories = []
all_point_predictions = []
all_smooth_predictions = []
models = []

for index in range(number_of_replicas):
    replica_number = index + 1
    print(f"[INFO]: Now training replica #{replica_number}")

    tf.keras.backend.clear_session()
    gc.collect()

    dnn_model = SurrogateModel(
        cross_section_symmetry_weight = 0.0,
        bsa_symmetry_weight = 0.0)
    dnn_model.compile(
        optimizer = tf.keras.optimizers.Adam(),
        loss = MSELoss())

    dnn_model_history = dnn_model.fit(
        x_training, y_training,
        validation_data = (x_validation, y_validation),
        epochs = NUMBER_OF_EPOCHS, batch_size = len(x_training),
        verbose = 0)
    
    all_histories.append(dnn_model_history.history)

    model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
    print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

    dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
    model_testing_loss = dictionary_of_keras_metrics["loss"]
    print(f"[INFO]: Test loss for replica #{replica_number}: {model_testing_loss}")

    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(dnn_model_history.history['loss'], 
        label = "Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(dnn_model_history.history['val_loss'], 
        label = "Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Loss", fontsize = 14.)
    axis.set_title(
        rf"Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
        fontsize = 14.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = f"./plots/surrogate_lc_replica_{replica_number}_v1.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(figure)

    del figure
    del axis
    
    # the output of the DNN is these standardized predictions: we need to inverse-transform:
    standardized_predictions = dnn_model.predict(features)
    replica_point_predictions = inverse_transform_labels(standardized_predictions)
    all_point_predictions.append(replica_point_predictions)

    # dnn_model.save(f"replica_{replica_number}_v1.keras")
    models.append(dnn_model)

all_point_predictions = np.array(all_point_predictions)

## (6): DNN Prediction Analysis:

### (6.1): Group kinematic settings by unique ($x_{\text{B}}$, $t$, and $Q^{2}$):

In [ ]:
grouped = test_dataframe.groupby(['t', 'x_b', 'q_squared'])

### (6.2): Make 2D plots showing DNN interpolation vs. original dataset for $d^{4}\sigma^{UU}$ and $\text{BSA}$:

In [ ]:
average_prediction = np.mean(all_point_predictions, axis = 0)
standard_dev_prediction = np.std(all_point_predictions, axis = 0)

phi_smooth = np.linspace(-1.0*np.pi, 1.0*np.pi, 361)

for (t_value, xb_value, qsquared_value), group in grouped:
    print(f"[INFO]: Processing t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values('phi')

    xsec_err = group['unp_beam_unp_target_xsec_err'].values
    bsa_err = group['unp_target_bsa_err'].values

    indices = group.index.values

    # discrete predictions:
    xsec_pred = average_prediction[indices, 0]
    bsa_pred = average_prediction[indices, 1]
    xsec_std = standard_dev_prediction[indices, 0]
    bsa_std = standard_dev_prediction[indices, 1]

    # smooth grid:
    smooth_dataframe = pd.DataFrame({
        "t": np.full_like(phi_smooth, t_value),
        "x_b": np.full_like(phi_smooth, xb_value),
        "q_squared": np.full_like(phi_smooth, qsquared_value),
        "phi": phi_smooth
    })

    smooth_dataframe["minus_t"] = -smooth_dataframe["t"]

    # transform into DNN feature space
    x_smooth_features = build_features(smooth_dataframe)

    smooth_preds_all = np.array([ inverse_transform_labels(model.predict(x_smooth_features, verbose = 0)) for model in models ])

    smooth_mean = np.mean(smooth_preds_all, axis = 0)
    smooth_std = np.std(smooth_preds_all, axis = 0)

    xsec_smooth_mean = smooth_mean[:, 0]
    bsa_smooth_mean = smooth_mean[:, 1]

    xsec_smooth_std = smooth_std[:, 0]
    bsa_smooth_std = smooth_std[:, 1]

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res = bsa_actual - bsa_pred

    chi2_xsec = np.sum(xsec_res**2) / len(phi)
    chi2_bsa = np.sum(bsa_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 2, figsize = (10, 8), sharex = 'col', layout = "tight")

    axes[1, 0].text(
        -0.1, -0.1, 
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axes[1, 0].transAxes)

    axes[0, 0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[0, 0].fill_between(
        phi_smooth, xsec_smooth_mean - xsec_smooth_std, xsec_smooth_mean + xsec_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    axes[0, 0].errorbar(
        phi, xsec_actual, yerr = xsec_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    axes[0, 0].set_ylabel(r"$d^{4}\sigma$ [nb / GeV$^{4}$]", fontsize = 14)
    axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle=':', alpha = 0.6)

    axes[0, 1].scatter(phi, xsec_res, color = 'blue', alpha = 0.6)
    axes[0, 1].axhline(0, color = 'black', linestyle = '--')
    axes[0, 1].set_title("Residuals")
    axes[0, 1].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 0].plot(phi_smooth, bsa_smooth_mean, color = 'green', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[1, 0].fill_between(
        phi_smooth, bsa_smooth_mean - bsa_smooth_std, bsa_smooth_mean + bsa_smooth_std,
        color = 'green', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    axes[1, 0].errorbar(
        phi, bsa_actual, yerr = bsa_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    
    axes[1, 0].set_ylabel("BSA", fontsize = 14)
    axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {chi2_bsa:.3f}$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 1].scatter(phi, bsa_res, color = 'purple', alpha = 0.6)
    axes[1, 1].axhline(0, color = 'black', linestyle = '--')
    axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 1].set_title("Residuals")
    axes[1, 1].grid(True, linestyle = ':', alpha = 0.6)
    
    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$t = {t_value:.3f}$, $x_\textrm{{B}} = {xb_value:.3f}$, $Q^2 = {qsquared_value:.3f}$",
        fontsize = 16
    )

    filename = f"./plots/t{t_value:.3f}_xb{xb_value:.3f}_q2{qsquared_value:.3f}_residuals_v1"
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(residuals_figure)

In [ ]:
xb_q2_groups = test_dataframe.groupby(['x_b', 'q_squared'])
n_xb_q2 = xb_q2_groups.ngroups
print(n_xb_q2)

In [ ]:
phi_grid = np.linspace(-np.pi, np.pi, 361)

In [ ]:
for (xb_value, qsquared_value), group in xb_q2_groups:

    group = group.sort_values(['t', 'phi'])

    t_values = np.sort(group['t'].unique())

    phi_meshgrid, t_meshgrid = np.meshgrid(phi_grid, t_values)

    phi_data = group['phi'].values
    t_data = group['t'].values

    indices = group.index.values

    xsec_pred = average_prediction[indices, 0]
    bsa_pred = average_prediction[indices, 1]

    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res = bsa_actual - bsa_pred

    colors_xsec = np.where(xsec_res >= 0, 'red', 'blue')
    colors_bsa = np.where(bsa_res >= 0, 'red', 'blue')

    surface_dataframe = pd.DataFrame({
        "t": t_meshgrid.ravel(),
        "x_b": np.full(t_meshgrid.size, xb_value),
        "q_squared": np.full(t_meshgrid.size, qsquared_value),
        "phi": phi_meshgrid.ravel()
    })

    surface_dataframe["minus_t"] = -surface_dataframe["t"]

    surface_features = build_features(surface_dataframe)

    surface_preds_standardized = np.array([ model.predict(surface_features, verbose = 0) for model in models ])
    surface_preds_all = np.array([ inverse_transform_labels(preds) for preds in surface_preds_standardized ])
    
    surface_mean = np.mean(surface_preds_all, axis = 0)
    surface_std_dev = np.std(surface_preds_all, axis = 0)

    xsec_surface = surface_mean[:, 0].reshape(t_meshgrid.shape)
    bsa_surface = surface_mean[:, 1].reshape(t_meshgrid.shape)

    xsec_stddev_surface = surface_std_dev[:, 0].reshape(t_meshgrid.shape)
    bsa_stddev_surface = surface_std_dev[:, 1].reshape(t_meshgrid.shape)

    zero_plane_xsec = np.zeros_like(xsec_surface)
    zero_plane_bsa = np.zeros_like(bsa_surface)

    fig = plt.figure(figsize = (10, 10), layout = "tight")

    ax1 = fig.add_subplot(2, 2, 1, projection = '3d')
    ax2 = fig.add_subplot(2, 2, 2, projection = '3d')
    ax3 = fig.add_subplot(2, 2, 3, projection = '3d')
    ax4 = fig.add_subplot(2, 2, 4, projection = '3d')

    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface, cmap = 'viridis', alpha = 0.5)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface + xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface - xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.scatter(phi_data, t_data, xsec_actual, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax1.set_xlabel(r'$\phi$ (Radians)')
    ax1.set_ylabel(r'$t$')
    ax1.set_zlabel(r'$d^{4}\sigma^{UU}$ [nb / GeV$^{4}$]')
    ax1.set_title('Cross Section', fontsize = 16)

    ax2.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_xsec, color = 'gray', alpha = 0.15)
    ax2.scatter(phi_data, t_data, xsec_res, color = colors_xsec, s = 20)
 
    ax2.set_xlabel(r'$\phi$')
    ax2.set_ylabel(r'$t$ [GeV$^{2}$]')
    ax2.set_zlabel('Residuals')
    ax2.set_title('Cross Section Residuals', fontsize = 16)

    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface, cmap='plasma', alpha = 0.5)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface + bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface - bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.scatter(phi_data, t_data, bsa_actual, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax3.set_xlabel(r'$\phi$ (Radians)')
    ax3.set_ylabel(r'$t$ [GeV$^{2}$]')
    ax3.set_zlabel('BSA')
    ax3.set_title('BSA', fontsize = 16)

    ax4.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_bsa, color = 'gray', alpha = 0.15)
    ax4.scatter(phi_data, t_data, bsa_res, color = colors_bsa, s = 20)

    ax4.set_xlabel(r'$\phi$')
    ax4.set_ylabel(r'$t$ [GeV$^{2}$]')
    ax4.set_zlabel('Residuals')
    ax4.set_title('BSA Residuals', fontsize = 16)

    fig.suptitle(
        r"DNN Interpolations Across $t$ and $\phi$"
        "\n"
        rf"Kinematic Setting: $x_\textrm{{B}} = {xb_value:.4g}$, $Q^2 = {qsquared_value:.4g}$",
        fontsize = 16)

    plot_filename = f"./plots/surface_xb{xb_value:.4g}_q2{qsquared_value:.4g}_v1"
    
    for extension in ['png', 'eps']:
        fig.savefig(f"{plot_filename}.{extension}", facecolor = 'white')

    plt.close(fig)

    # cleanup:
    del fig
    del ax1
    del ax2
    del ax3
    del ax4